# splitQP

This notebook solves a family of quadratic programs that share a fixed `P` and `A`
while `q`, `l`, and `u` vary. `solve_batch` treats the members as independent;
`solve_sequence` follows the ordered family with warm starts. Both reuse the single
Cholesky factorization formed when the `Solver` is constructed.

In [1]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import splitqp

In [2]:
# One fixed (P, A); member i tracks a moving centre x(t_i), so adjacent members
# have nearby optima (good for warm starts).
rng = np.random.default_rng(0)
n, m, B = 8, 12, 24
M = rng.normal(size=(n, n))
P = M @ M.T + n * np.eye(n)                    # fixed, symmetric positive definite
A = rng.normal(size=(m, n)); A /= np.linalg.norm(A, axis=1, keepdims=True)

x0, dx = rng.normal(size=n), rng.normal(size=n)
width = rng.uniform(0.3, 0.8, size=m)
t = np.linspace(0.0, 1.0, B)                   # the ordered family parameter
qs = np.zeros((B, n)); ls = np.zeros((B, m)); us = np.zeros((B, m))
for i, ti in enumerate(t):
    xc = x0 + ti * dx
    c = A @ xc
    ls[i], us[i] = c - width, c + width
    qs[i] = -(P @ xc)

print(f"variables:      {n}")
print(f"constraints:    {m}")
print(f"family members: {B}")

variables:      8
constraints:    12
family members: 24


In [3]:
solver = splitqp.Solver(P, A)                        # the one factorization
scalar = solver.solve(qs[0], ls[0], us[0])           # one member
batch = solver.solve_batch(qs, ls, us)               # independent members
seq = solver.solve_sequence(qs, ls, us)              # ordered, warm-started members

xb, xq = np.asarray(batch.x), np.asarray(seq.x)
b_solved = int(np.sum(np.asarray(batch.status) == "solved"))
q_solved = int(np.sum(np.asarray(seq.status) == "solved"))

print(f"scalar status:          {scalar.status}")
print(f"batch solved:           {b_solved}/{B}")
print(f"sequence solved:        {q_solved}/{B}")
print(f"factorizations:         {solver.factorizations}")
print(f"max scalar-vs-batch:    {np.max(np.abs(np.asarray(scalar.x) - xb[0])):.2e}")
print(f"max batch-vs-sequence:  {np.max(np.abs(xb - xq)):.2e}")
# batch and sequence stop at a relative 1e-6 residual from different warm starts,
# so they agree to the solve tolerance rather than bit-for-bit.  This is not an error.

scalar status:          solved
batch solved:           24/24
sequence solved:        24/24
factorizations:         1
max scalar-vs-batch:    2.22e-16
max batch-vs-sequence:  4.00e-06


In [4]:
# A few representative members (first, middle, last), not the whole family.
seq_iters = np.asarray(seq.iterations)
print(f"{'member':>6} {'parameter':>10} {'x[0]':>10} {'x[1]':>10} {'seq_iters':>10}")
for i in (0, B // 2, B - 1):
    print(f"{i:>6} {t[i]:>10.3f} {xq[i, 0]:>10.4f} {xq[i, 1]:>10.4f} {int(seq_iters[i]):>10}")

member  parameter       x[0]       x[1]  seq_iters
     0      0.000     0.5131    -0.2976         27
    12      0.522     1.3725     0.1811         20
    23      1.000     2.1604     0.6199         20


In [5]:
batch_iters = np.asarray(batch.iterations)
print("sequence iterations:  "
      f"min {int(seq_iters.min())}, median {int(np.median(seq_iters))}, max {int(seq_iters.max())}")
print("batch iterations:     "
      f"min {int(batch_iters.min())}, median {int(np.median(batch_iters))}, max {int(batch_iters.max())}")

sequence iterations:  min 20, median 20, max 27
batch iterations:     min 27, median 27, max 28


`solve_batch` members are independent; `solve_sequence` members are ordered and
warm-started, which is why the sequence usually takes fewer total iterations. Both
reuse the same construction-time factorization. For optional local timing
comparisons, see `bench.py`.